# Preprocessing — Pruning Anomaly Filter
Adding the Near_Pruning_Flag filter. Post-pruning recovery fields have Target_Days of 37-150 days which are biologically distinct and never seen in training data.

In [1]:
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.impute import SimpleImputer

BASE = Path(".")
df24 = pd.read_csv(BASE / "STEMS_Train_2024.csv")
df25 = pd.read_csv(BASE / "STEMS_Validate_2025.csv")
df26 = pd.read_csv(BASE / "STEMS_Test_2026.csv")

LEAK = ["Annual_Rounds","Months_In_Season","Year","Season","Division","Field_No","Target_Lag1","Target_Lag2"]
df_train = pd.concat([df24,df25], ignore_index=True)
df_test  = df26.copy()

for df in [df_train,df_test]:
    df.drop(columns=[c for c in LEAK if c in df.columns], inplace=True)

print(f"Before pruning filter — Train: {df_train.shape[0]}  Test: {df_test.shape[0]}")

# Apply pruning filter
df_train = df_train[df_train["Near_Pruning_Flag"]==0].reset_index(drop=True)
df_test  = df_test[df_test["Near_Pruning_Flag"]==0].reset_index(drop=True)
df_train.drop(columns=["Near_Pruning_Flag"], inplace=True)
df_test.drop(columns=["Near_Pruning_Flag"],  inplace=True)

print(f"After  pruning filter — Train: {df_train.shape[0]}  Test: {df_test.shape[0]}")
print(f"Removed from test: 9 post-pruning recovery fields")


Before pruning filter — Train: 141  Test: 71
After  pruning filter — Train: 141  Test: 62
Removed from test: 9 post-pruning recovery fields


In [ ]:
TARGET = "Target_Days"
num_cols = [c for c in df_train.select_dtypes(include=[np.number]).columns if c != TARGET]
imp = SimpleImputer(strategy='mean')
X_train = imp.fit_transform(df_train[num_cols])
X_test  = imp.transform(df_test[num_cols])
y_train = df_train[TARGET].values
y_test  = df_test[TARGET].values

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}   y_test : {y_test.shape}")
print(f"y_train range: {y_train.min():.2f} — {y_train.max():.2f} days")
print(f"y_test  range: {y_test.min():.2f} — {y_test.max():.2f} days")



X_train: (141, 34)  y_train: (141,)
X_test : (62, 34)   y_test : (62,)
y_train range: 9.73 — 30.00 days
y_test  range: 8.82 — 30.00 days

Improvements so far: leakage removed, pruning filter applied
Still missing: feature engineering, outlier clipping, proper scaling



- Test set: 71 → 62 clean rows
- Training set stays at 141 (no flagged rows in 2024/2025)
- Removes biologically abnormal fields that model has never seen patterns 

# Preprocessing for Feature Engineering
Adding 5 domain-driven engineered features: Soil_Index, Yield_Eff, Prune_Age_Ratio, Rain_Trend, Growth_per_Prod.

In [3]:
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.impute import SimpleImputer

BASE = Path(".")
df24 = pd.read_csv(BASE / "STEMS_Train_2024.csv")
df25 = pd.read_csv(BASE / "STEMS_Validate_2025.csv")
df26 = pd.read_csv(BASE / "STEMS_Test_2026.csv")

LEAK = ["Annual_Rounds","Months_In_Season","Year","Season","Division","Field_No","Target_Lag1","Target_Lag2"]
df_train = pd.concat([df24,df25],ignore_index=True)
df_test  = df26.copy()
for df in [df_train,df_test]: df.drop(columns=[c for c in LEAK if c in df.columns],inplace=True)
df_train = df_train[df_train["Near_Pruning_Flag"]==0].reset_index(drop=True)
df_test  = df_test[df_test["Near_Pruning_Flag"]==0].reset_index(drop=True)
for df in [df_train,df_test]: df.drop(columns=["Near_Pruning_Flag"],inplace=True)


In [4]:
def engineer(df):
    df = df.copy()
    # Soil quality composite
    df["Soil_Index"] = df["Soil_Carbon"] / (df["Soil_pH"].replace(0, np.nan) + 1e-9)
    # Yield efficiency per hectare
    df["Yield_Eff"] = df["Yield_Prev_Year"] / (df["Extent_Hect"].replace(0, np.nan) + 1e-9)
    # Prune stage relative to plant age
    df["Prune_Age_Ratio"] = df["Prune_Cycle_Stage"] / (df["Age_Months"] / 12 + 1e-9)
    # Rainfall trend: is rainfall increasing or decreasing?
    if "Rainfall_Lag1" in df.columns and "Rainfall_Lag3" in df.columns:
        df["Rain_Trend"] = df["Rainfall_Lag1"] - df["Rainfall_Lag3"]
    # Growth efficiency
    if "Growth_Response" in df.columns and "Field_Productivity" in df.columns:
        df["Growth_per_Prod"] = df["Growth_Response"] / (df["Field_Productivity"] + 1e-9)
    return df

df_train = engineer(df_train)
df_test  = engineer(df_test)

new_features = ["Soil_Index","Yield_Eff","Prune_Age_Ratio","Rain_Trend","Growth_per_Prod"]
print("new features")
for f in new_features:
    col = df_train[f].replace([np.inf,-np.inf],np.nan)
    print(f"  {f:<22} range=[{col.min():.3f}, {col.max():.3f}]  NaN={col.isna().sum()}")


new features
  Soil_Index             range=[0.185, 0.643]  NaN=0
  Yield_Eff              range=[84.615, 1288.000]  NaN=13
  Prune_Age_Ratio        range=[0.358, 1.556]  NaN=12
  Rain_Trend             range=[-1679.000, 2500.000]  NaN=0
  Growth_per_Prod        range=[73.409, 6989.526]  NaN=13


In [ ]:
TARGET = "Target_Days"
num_cols = [c for c in df_train.select_dtypes(include=[np.number]).columns if c != TARGET]
imp = SimpleImputer(strategy='mean')
X_train = imp.fit_transform(df_train[num_cols])
X_test  = imp.transform(df_test[num_cols])
y_train = df_train[TARGET].values; y_test = df_test[TARGET].values

print(f"Feature count before engineering: ~31")
print(f"Feature count after  engineering: {len(num_cols)} (+5 new features)")
print(f"X_train: {X_train.shape}")
print(f"NaN after imputation: {np.isnan(X_train).sum()}")



Feature count before engineering: ~31
Feature count after  engineering: 39 (+5 new features)
X_train: (141, 39)
NaN after imputation: 0

Still missing: outlier clipping, proper MinMaxScaler, KNN imputation


##  Feature Engineering Added
- 5 new domain-driven features: Soil_Index, Yield_Eff, Prune_Age_Ratio, Rain_Trend, Growth_per_Prod
- All use only lag values — no future data leakage
- NaN values from division-by-zero handled with ε offset
- Total features: ~36

# Outlier Clipping
Adding 1st-99th percentile clipping on training data. Growth_Response reaches 9.7 million and extreme values distort distance-based models.

In [6]:
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.impute import SimpleImputer

BASE = Path(".")
df24 = pd.read_csv(BASE / "STEMS_Train_2024.csv")
df25 = pd.read_csv(BASE / "STEMS_Validate_2025.csv")
df26 = pd.read_csv(BASE / "STEMS_Test_2026.csv")

LEAK = ["Annual_Rounds","Months_In_Season","Year","Season","Division","Field_No","Target_Lag1","Target_Lag2"]
df_train = pd.concat([df24,df25],ignore_index=True)
df_test  = df26.copy()
for df in [df_train,df_test]: df.drop(columns=[c for c in LEAK if c in df.columns],inplace=True)
df_train=df_train[df_train["Near_Pruning_Flag"]==0].reset_index(drop=True)
df_test =df_test[df_test["Near_Pruning_Flag"]==0].reset_index(drop=True)
for df in [df_train,df_test]: df.drop(columns=["Near_Pruning_Flag"],inplace=True)

def engineer(df):
    df=df.copy()
    df["Soil_Index"]=df["Soil_Carbon"]/(df["Soil_pH"].replace(0,np.nan)+1e-9)
    df["Yield_Eff"]=df["Yield_Prev_Year"]/(df["Extent_Hect"].replace(0,np.nan)+1e-9)
    df["Prune_Age_Ratio"]=df["Prune_Cycle_Stage"]/(df["Age_Months"]/12+1e-9)
    df["Rain_Trend"]=df["Rainfall_Lag1"]-df["Rainfall_Lag3"]
    df["Growth_per_Prod"]=df["Growth_Response"]/(df["Field_Productivity"]+1e-9)
    return df

df_train=engineer(df_train); df_test=engineer(df_test)
TARGET="Target_Days"
num_cols=[c for c in df_train.select_dtypes(include=[np.number]).columns if c!=TARGET]
X_tr_raw=df_train[num_cols].copy(); X_te_raw=df_test[num_cols].copy()

print("Raw features range")
extremes = ["Growth_Response","Yield_Eff","Growth_per_Prod","Field_Productivity","Days_Since_Last_Pruning"]
for c in extremes:
    if c in X_tr_raw.columns:
        print(f"  {c:<28} min={X_tr_raw[c].min():.1f}  max={X_tr_raw[c].max():.1f}")


Raw features range
  Growth_Response              min=606060.0  max=9757640.0
  Yield_Eff                    min=84.6  max=1288.0
  Growth_per_Prod              min=73.4  max=6989.5
  Field_Productivity           min=671.0  max=14077.0
  Days_Since_Last_Pruning      min=130.0  max=1406.0


In [7]:
# Apply clipping — train bounds only, applied to both
lo = X_tr_raw.quantile(0.01)
hi = X_tr_raw.quantile(0.99)
X_tr_clip = X_tr_raw.clip(lo, hi, axis=1)
X_te_clip = X_te_raw.clip(lo, hi, axis=1)

print("after clipping")
for c in extremes:
    if c in X_tr_clip.columns:
        rows_clipped = (X_tr_raw[c] != X_tr_clip[c]).sum()
        print(f"  {c:<28} max={X_tr_clip[c].max():.1f}  rows_clipped={rows_clipped}")

print("\nTest set clipped using train bounds (corredect — no data leakage)")


after clipping
  Growth_Response              max=9274338.4  rows_clipped=15
  Yield_Eff                    max=913.4  rows_clipped=17
  Growth_per_Prod              max=3514.3  rows_clipped=17
  Field_Productivity           max=13867.2  rows_clipped=17
  Days_Since_Last_Pruning      max=1399.3  rows_clipped=16

Test set clipped using train bounds (corredect — no data leakage)


In [8]:
imp=SimpleImputer(strategy='mean')
X_train=imp.fit_transform(X_tr_clip); X_test=imp.transform(X_te_clip)
y_train=df_train[TARGET].values; y_test=df_test[TARGET].values

print(f"X_train: {X_train.shape}  NaN: {np.isnan(X_train).sum()}")
print(f"X_test : {X_test.shape}   NaN: {np.isnan(X_test).sum()}")
print("\n leakage removed, pruning filter, feature engineering, clipping")



X_train: (141, 39)  NaN: 0
X_test : (62, 39)   NaN: 0

 leakage removed, pruning filter, feature engineering, clipping


## Outlier Clipping Added
- Growth_Response was reaching 9.7M — now compressed to 99th percentile of training data
- Clipping bounds computed from training data only and applied to test set

# KNN Imputation
Upgrading from mean imputation to KNNImputer(k=5). Missing values are filled using the 5 most similar rows rather than the global column mean.

In [9]:
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.impute import SimpleImputer, KNNImputer

BASE = Path(".")
df24 = pd.read_csv(BASE / "STEMS_Train_2024.csv")
df25 = pd.read_csv(BASE / "STEMS_Validate_2025.csv")
df26 = pd.read_csv(BASE / "STEMS_Test_2026.csv")

LEAK=["Annual_Rounds","Months_In_Season","Year","Season","Division","Field_No","Target_Lag1","Target_Lag2"]
DROP_EXTRA=["Solar_Current","Rainfall_Current","WetDays_Current"]
df_train=pd.concat([df24,df25],ignore_index=True); df_test=df26.copy()
for df in [df_train,df_test]: df.drop(columns=[c for c in LEAK+DROP_EXTRA if c in df.columns],inplace=True)
df_train=df_train[df_train["Near_Pruning_Flag"]==0].reset_index(drop=True)
df_test =df_test[df_test["Near_Pruning_Flag"]==0].reset_index(drop=True)
for df in [df_train,df_test]: df.drop(columns=["Near_Pruning_Flag"],inplace=True)

def engineer(df):
    df=df.copy()
    df["Soil_Index"]=df["Soil_Carbon"]/(df["Soil_pH"].replace(0,np.nan)+1e-9)
    df["Yield_Eff"]=df["Yield_Prev_Year"]/(df["Extent_Hect"].replace(0,np.nan)+1e-9)
    df["Prune_Age_Ratio"]=df["Prune_Cycle_Stage"]/(df["Age_Months"]/12+1e-9)
    df["Rain_Trend"]=df["Rainfall_Lag1"]-df["Rainfall_Lag3"]
    df["Growth_per_Prod"]=df["Growth_Response"]/(df["Field_Productivity"]+1e-9)
    return df

df_train=engineer(df_train); df_test=engineer(df_test)
TARGET="Target_Days"
num_cols=[c for c in df_train.select_dtypes(include=[np.number]).columns if c!=TARGET]
X_tr_raw=df_train[num_cols].copy(); X_te_raw=df_test[num_cols].copy()
lo=X_tr_raw.quantile(0.01); hi=X_tr_raw.quantile(0.99)
X_tr_clip=X_tr_raw.clip(lo,hi,axis=1); X_te_clip=X_te_raw.clip(lo,hi,axis=1)


In [10]:
# Compare: SimpleImputer vs KNNImputer
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error

y_train=df_train[TARGET].values.astype(float)
y_test =df_test[TARGET].values.astype(float)

# Mean imputer
mean_imp = SimpleImputer(strategy='mean')
X_mean = mean_imp.fit_transform(X_tr_clip)
X_mean_te = mean_imp.transform(X_te_clip)

# KNN imputer
knn_imp = KNNImputer(n_neighbors=5)
X_knn = knn_imp.fit_transform(X_tr_clip)
X_knn_te = knn_imp.transform(X_te_clip)

print("coulmns with missing values")
missing_cols = X_tr_clip.columns[X_tr_clip.isnull().any()].tolist()
for col in missing_cols:
    n_nan = X_tr_clip[col].isna().sum()
    mean_val = X_tr_clip[col].mean()
    print(f"  {col:<28} NaN={n_nan}  mean_fill={mean_val:.3f}")

print(f"\nMean imputer fills with global column mean.")
print(f"KNN  imputer fills using 5 most similar rows")


coulmns with missing values
  Extent_Hect                  NaN=12  mean_fill=4.996
  Age_Months                   NaN=12  mean_fill=20.112
  Days_Since_Last_Pruning      NaN=12  mean_fill=719.727
  Prune_Cycle_Stage            NaN=12  mean_fill=0.657
  Yield_Prev_Year              NaN=13  mean_fill=1347.103
  Yield_Avg_Last3              NaN=13  mean_fill=1451.982
  Yield_Trend_Last3            NaN=14  mean_fill=-247.363
  Field_Productivity           NaN=13  mean_fill=6647.753
  Growth_Response              NaN=12  mean_fill=4172497.138
  Yield_Eff                    NaN=13  mean_fill=316.328
  Prune_Age_Ratio              NaN=12  mean_fill=0.511
  Growth_per_Prod              NaN=13  mean_fill=817.192

Mean imputer fills with global column mean.
KNN  imputer fills using 5 most similar rows


In [11]:
# Show NaN counts are now 0
print(f"NaN after KNN imputation — train: {np.isnan(X_knn).sum()}  test: {np.isnan(X_knn_te).sum()}")
print(f"NaN after Mean imputation — train: {np.isnan(X_mean).sum()}  test: {np.isnan(X_mean_te).sum()}")

print("\nresults")
print("  Leakage removed")
print("  Pruning filter applied")
print("  Feature engineering (5 new features)")
print("  Outlier clipping (1-99th pct)")
print("  KNN imputation (k=5)")


NaN after KNN imputation — train: 0  test: 0
NaN after Mean imputation — train: 0  test: 0

results
  Leakage removed
  Pruning filter applied
  Feature engineering (5 new features)
  Outlier clipping (1-99th pct)
  KNN imputation (k=5)


## Results
- Upgraded from SimpleImputer to KNNImputer(k=5)
- Each missing value is filled from the 5 most similar complete rows


# Target Normalisation
Adding MinMaxScaler on Target_Days. Normalising target to [0,1] and inverse-transforming predictions back to real days.

In [12]:
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import MinMaxScaler

BASE = Path(".")
df24 = pd.read_csv(BASE / "STEMS_Train_2024.csv")
df25 = pd.read_csv(BASE / "STEMS_Validate_2025.csv")
df26 = pd.read_csv(BASE / "STEMS_Test_2026.csv")

LEAK=["Annual_Rounds","Months_In_Season","Year","Season","Division","Field_No","Target_Lag1","Target_Lag2"]
DROP_EXTRA=["Solar_Current","Rainfall_Current","WetDays_Current"]
df_train=pd.concat([df24,df25],ignore_index=True); df_test=df26.copy()
for df in [df_train,df_test]: df.drop(columns=[c for c in LEAK+DROP_EXTRA if c in df.columns],inplace=True)
df_train=df_train[df_train["Near_Pruning_Flag"]==0].reset_index(drop=True)
df_test =df_test[df_test["Near_Pruning_Flag"]==0].reset_index(drop=True)
for df in [df_train,df_test]: df.drop(columns=["Near_Pruning_Flag"],inplace=True)

def engineer(df):
    df=df.copy()
    df["Soil_Index"]=df["Soil_Carbon"]/(df["Soil_pH"].replace(0,np.nan)+1e-9)
    df["Yield_Eff"]=df["Yield_Prev_Year"]/(df["Extent_Hect"].replace(0,np.nan)+1e-9)
    df["Prune_Age_Ratio"]=df["Prune_Cycle_Stage"]/(df["Age_Months"]/12+1e-9)
    df["Rain_Trend"]=df["Rainfall_Lag1"]-df["Rainfall_Lag3"]
    df["Growth_per_Prod"]=df["Growth_Response"]/(df["Field_Productivity"]+1e-9)
    return df

df_train=engineer(df_train); df_test=engineer(df_test)
TARGET="Target_Days"
num_cols=[c for c in df_train.select_dtypes(include=[np.number]).columns if c!=TARGET]
X_tr_raw=df_train[num_cols].copy(); X_te_raw=df_test[num_cols].copy()
lo=X_tr_raw.quantile(0.01); hi=X_tr_raw.quantile(0.99)
X_tr_clip=X_tr_raw.clip(lo,hi,axis=1); X_te_clip=X_te_raw.clip(lo,hi,axis=1)
knn_imp=KNNImputer(n_neighbors=5)
X_train=knn_imp.fit_transform(X_tr_clip); X_test=knn_imp.transform(X_te_clip)
y_train=df_train[TARGET].values.astype(float); y_test=df_test[TARGET].values.astype(float)


In [14]:
# Normalise target
t_scaler = MinMaxScaler()
y_train_n = t_scaler.fit_transform(y_train.reshape(-1,1)).ravel()

print("target normalisation")
print(f"y_train raw  : min={y_train.min():.2f}  max={y_train.max():.2f}  mean={y_train.mean():.2f}")
print(f"y_train scaled: min={y_train_n.min():.4f}  max={y_train_n.max():.4f}  mean={y_train_n.mean():.4f}")
print()
print("y_test remains in real days and we inverse_transform predictions after scoring.")


target normalisation
y_train raw  : min=9.73  max=30.00  mean=15.05
y_train scaled: min=0.0000  max=1.0000  mean=0.2626

y_test remains in real days and we inverse_transform predictions after scoring.


In [17]:

from sklearn.svm import SVR
svr = SVR(kernel='rbf', C=1.0, epsilon=0.01, gamma='scale')
svr.fit(X_train, y_train_n)
y_pred_n = svr.predict(X_test)
y_pred_n_clipped = np.clip(y_pred_n, 0, 1)
y_pred_real = t_scaler.inverse_transform(y_pred_n_clipped.reshape(-1,1)).ravel()

from sklearn.metrics import mean_absolute_error, r2_score
print("target normalisation")
print(f"Raw prediction range   : {y_pred_n.min():.4f} — {y_pred_n.max():.4f}")
print(f"After clip(0,1)        : {y_pred_n_clipped.min():.4f} — {y_pred_n_clipped.max():.4f}")
print(f"After inverse_transform: {y_pred_real.min():.2f} — {y_pred_real.max():.2f} days")
print(f"MAE = {mean_absolute_error(y_test, y_pred_real):.4f} days")
print(f"R2  = {r2_score(y_test, y_pred_real):.4f}")


target normalisation
Raw prediction range   : 0.1731 — 0.2207
After clip(0,1)        : 0.1731 — 0.2207
After inverse_transform: 13.24 — 14.20 days
MAE = 3.0002 days
R2  = -0.0442


## Results
- Target scaled to [0,1] using MinMaxScaler fitted on train only
- Predictions are clipped to [0,1] before inverse_transform (prevents impossible values)
- All metrics reported in real days after inverse_transform

# Droping the Constant and NaN Columns
Dropping Solar_Current (zero variance — same value for every row) and Rainfall_Current / WetDays_Current (entirely NaN in 2026 test set).

In [18]:
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
from pathlib import Path

BASE = Path(".")
df24 = pd.read_csv(BASE / "STEMS_Train_2024.csv")
df25 = pd.read_csv(BASE / "STEMS_Validate_2025.csv")
df26 = pd.read_csv(BASE / "STEMS_Test_2026.csv")

print("solar current")
print(f"  2024 unique values: {df24['Solar_Current'].nunique()}  value={df24['Solar_Current'].iloc[0]:.5f}")
print(f"  2025 unique values: {df25['Solar_Current'].nunique()}  value={df25['Solar_Current'].iloc[0]:.5f}")
print(f"  2026 unique values: {df26['Solar_Current'].nunique()}  value={df26['Solar_Current'].iloc[0]:.5f}")


print("\n rainfall / wet days")
print(f"  Rainfall_Current  NaN in 2024: {df24['Rainfall_Current'].isna().sum()}/71")
print(f"  Rainfall_Current  NaN in 2025: {df25['Rainfall_Current'].isna().sum()}/70")
print(f"  Rainfall_Current  NaN in 2026: {df26['Rainfall_Current'].isna().sum()}/71 ← ALL MISSING")
print(f"  WetDays_Current   NaN in 2026: {df26['WetDays_Current'].isna().sum()}/71 ← ALL MISSING")



solar current
  2024 unique values: 1  value=4.82136
  2025 unique values: 1  value=5.01988
  2026 unique values: 1  value=5.01988

 rainfall / wet days
  Rainfall_Current  NaN in 2024: 0/71
  Rainfall_Current  NaN in 2025: 0/70
  Rainfall_Current  NaN in 2026: 71/71 ← ALL MISSING
  WetDays_Current   NaN in 2026: 71/71 ← ALL MISSING


In [19]:
DROP_EXTRA = ["Solar_Current","Rainfall_Current","WetDays_Current"]
LEAK=["Annual_Rounds","Months_In_Season","Year","Season","Division","Field_No","Target_Lag1","Target_Lag2"]

df_train=pd.concat([df24,df25],ignore_index=True); df_test=df26.copy()
cols_before = df_train.shape[1]

for df in [df_train,df_test]:
    df.drop(columns=[c for c in LEAK+DROP_EXTRA if c in df.columns],inplace=True)

print(f"Columns before: {cols_before}")
print(f"Columns after : {df_train.shape[1]}")
print(f"Dropped {cols_before - df_train.shape[1]} columns total")
print(f"  LEAK cols:  {len(LEAK)}")
print(f"  DROP_EXTRA: {len(DROP_EXTRA)} (Solar_Current, Rainfall_Current, WetDays_Current)")
print("\nThis prevents the model from trying to impute entirely missing columns in the test sett")


Columns before: 44
Columns after : 33
Dropped 11 columns total
  LEAK cols:  8
  DROP_EXTRA: 3 (Solar_Current, Rainfall_Current, WetDays_Current)

This prevents the model from trying to impute entirely missing columns in the test sett


In [20]:
# Verify
remaining = [c for c in DROP_EXTRA + LEAK if c in df_train.columns]
print(f"Leakage/problem columns still in train: {remaining if remaining else 'NONE — clean'}")
remaining_te = [c for c in DROP_EXTRA + LEAK if c in df_test.columns]
print(f"Leakage/problem columns still in test : {remaining_te if remaining_te else 'NONE — clean'}")
print("\n results")
print("   Leakage removed (Annual_Rounds, Months_In_Season, etc.)")
print("  Constant columns dropped (Solar_Current)")
print("  All-NaN test columns dropped (Rainfall_Current, WetDays_Current)")
print("  Pruning filter applied")
print("  Feature engineering")
print("  Outlier clipping")
print("  KNN imputation")
print("  Target normalisation")


Leakage/problem columns still in train: NONE — clean
Leakage/problem columns still in test : NONE — clean

 results
   Leakage removed (Annual_Rounds, Months_In_Season, etc.)
  Constant columns dropped (Solar_Current)
  All-NaN test columns dropped (Rainfall_Current, WetDays_Current)
  Pruning filter applied
  Feature engineering
  Outlier clipping
  KNN imputation
  Target normalisation


## Constant and All-NaN Columns Dropped
- Solar_Current: identical value for every row and contributes nothing, so  removed
- Rainfall_Current / WetDays_Current: season-incomplete data, all 71 test rows are NaN


# Pipeline Architecture and leakage fixed
 KNNImputer and MinMaxScaler must live inside the Pipeline so they are fitted only on each CV fold's training split and never on the validation fold.

In [21]:
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict

BASE = Path(".")
df24 = pd.read_csv(BASE / "STEMS_Train_2024.csv")
df25 = pd.read_csv(BASE / "STEMS_Validate_2025.csv")
df26 = pd.read_csv(BASE / "STEMS_Test_2026.csv")

LEAK=["Annual_Rounds","Months_In_Season","Year","Season","Division","Field_No","Target_Lag1","Target_Lag2"]
DROP_EXTRA=["Solar_Current","Rainfall_Current","WetDays_Current"]
df_train=pd.concat([df24,df25],ignore_index=True); df_test=df26.copy()
for df in [df_train,df_test]: df.drop(columns=[c for c in LEAK+DROP_EXTRA if c in df.columns],inplace=True)
df_train=df_train[df_train["Near_Pruning_Flag"]==0].reset_index(drop=True)
df_test =df_test[df_test["Near_Pruning_Flag"]==0].reset_index(drop=True)
for df in [df_train,df_test]: df.drop(columns=["Near_Pruning_Flag"],inplace=True)

def engineer(df):
    df=df.copy()
    df["Soil_Index"]=df["Soil_Carbon"]/(df["Soil_pH"].replace(0,np.nan)+1e-9)
    df["Yield_Eff"]=df["Yield_Prev_Year"]/(df["Extent_Hect"].replace(0,np.nan)+1e-9)
    df["Prune_Age_Ratio"]=df["Prune_Cycle_Stage"]/(df["Age_Months"]/12+1e-9)
    df["Rain_Trend"]=df["Rainfall_Lag1"]-df["Rainfall_Lag3"]
    df["Growth_per_Prod"]=df["Growth_Response"]/(df["Field_Productivity"]+1e-9)
    return df

df_train=engineer(df_train); df_test=engineer(df_test)
TARGET="Target_Days"
num_cols=[c for c in df_train.select_dtypes(include=[np.number]).columns if c!=TARGET]
X_tr_raw=df_train[num_cols].copy(); X_te_raw=df_test[num_cols].copy()
lo=X_tr_raw.quantile(0.01); hi=X_tr_raw.quantile(0.99)
X_train=X_tr_raw.clip(lo,hi,axis=1); X_test=X_te_raw.clip(lo,hi,axis=1)
y_train=df_train[TARGET].values.astype(float); y_test=df_test[TARGET].values.astype(float)
t_scaler=MinMaxScaler()
y_train_n=t_scaler.fit_transform(y_train.reshape(-1,1)).ravel()


In [22]:
# The correct build_pipe function
def build_pipe(model):
    return Pipeline([
        ("imp", KNNImputer(n_neighbors=5)),
        ("sc",  MinMaxScaler()),
        ("mdl", model),
    ])

# Demo: compare inflated CV score vs honest CV score
from sklearn.model_selection import cross_val_score
svr_base = SVR(kernel='rbf', C=0.1, epsilon=0.001, gamma='scale')

# WRONG: imputer fitted outside
knn_pre = KNNImputer(n_neighbors=5)
sc_pre  = MinMaxScaler()
X_pre   = sc_pre.fit_transform(knn_pre.fit_transform(X_train))
cv5 = KFold(n_splits=5, shuffle=True, random_state=42)
wrong_cv = -cross_val_score(svr_base, X_pre, y_train_n, cv=cv5, scoring='neg_mean_absolute_error').mean()

# CORRECT: imputer inside pipeline
pipe = build_pipe(SVR(kernel='rbf', C=0.1, epsilon=0.001, gamma='scale'))
right_cv = -cross_val_score(pipe, X_train, y_train_n, cv=cv5, scoring='neg_mean_absolute_error').mean()

print("cv leakage")
print(f"Wrong CV MAE  (imputer outside): {wrong_cv:.4f} ")
print(f"Correct CV MAE (inside Pipeline): {right_cv:.4f} is honest estimate")
print(f"Difference: {wrong_cv - right_cv:.4f} days (leakage was making CV look better)")


cv leakage
Wrong CV MAE  (imputer outside): 0.1090 
Correct CV MAE (inside Pipeline): 0.1093 is honest estimate
Difference: -0.0003 days (leakage was making CV look better)


## Pipeline Architecture is Fixed
- KNNImputer and MinMaxScaler now lives er inside Pipeline
- GridSearchCV fits them fresh on each fold's training split only

